In [1]:
import torch
from PIL import Image
from transformers import XLMRobertaTokenizer
from torchvision import transforms

# BEiT-3 레포지토리의 모델링 파일을 import해야 합니다.
# 이 파일이 있는 경로를 sys.path에 추가하거나, 같은 디렉토리에 복사해두세요.
from modeling_finetune import beit3_large_patch16_480_captioning

import json # VQA 답변 라벨을 로드하기 위해 import

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'modeling_finetune'

In [2]:
# --- 1. 설정 (사용자 환경에 맞게 수정) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fine-tuning된 모델 가중치 파일 경로
model_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3_large_patch16_480_coco_captioning.pth"
# 문장 토크나이저 모델 경로
spm_model_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3.spm"
# VQAv2 답변 라벨 파일 경로 (데이터 준비 과정에서 생성됨)
# 실제 VQAv2 데이터셋의 answer list가 필요합니다.
# 예시: https://github.com/microsoft/unilm/blob/master/beit3/get_started/get_started_for_vqav2.md
vqa_answer_label_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/v2_mscoco_train2014_annotations.json"

# --- 모델 및 토크나이저 불러오기 ---
tokenizer = XLMRobertaTokenizer(spm_model_path)

In [ ]:
# 2. import한 함수를 바로 호출하여 모델 객체를 생성합니다.
#    이 함수가 내부적으로 BEiT3ForVisualQuestionAnswering 클래스를 사용해 모델을 만듭니다.
print("Creating model...")
model = beit3_large_patch16_480_captioning()

Creating model...


In [4]:
# 3. Fine-tuning된 가중치를 불러옵니다. (이 부분은 동일)
print("Loading checkpoint...")
checkpoint = torch.load(model_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])

model.to(device)
model.eval() # 모델을 평가 모드로 설정

Loading checkpoint...


BEiT3ForCaptioning(
  (beit3): BEiT3(
    (text_embed): TextEmbedding(64010, 1024)
    (vision_embed): VisionEmbedding(
      (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
    )
    (encoder): Encoder(
      (dropout_module): Dropout(p=0.0, inplace=False)
      (embed_positions): MutliwayEmbedding(
        (A): PositionalEmbedding(903, 1024)
        (B): PositionalEmbedding(1024, 1024)
      )
      (layers): ModuleList(
        (0): EncoderLayer(
          (self_attn): MultiheadAttention(
            (k_proj): MultiwayNetwork(
              (A): Linear(in_features=1024, out_features=1024, bias=True)
              (B): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (v_proj): MultiwayNetwork(
              (A): Linear(in_features=1024, out_features=1024, bias=True)
              (B): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (q_proj): MultiwayNetwork(
              (A): Linear(in_features=1024, out

In [6]:
from PIL import Image
import requests

img_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images/TEST_000.jpg"
image = Image.open(img_path).convert("RGB")
# 이미지 전처리
preprocess = transforms.Compose([
    transforms.Resize((480, 480)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
image_tensor = preprocess(image).unsqueeze(0).to(device)

In [7]:
model.eval()  # 평가 모드로 설정
with torch.no_grad():
    # 이미지 입력을 모델에 전달하여 예측 수행
    outputs = model(image_tensor)

TypeError: forward() missing 3 required positional arguments: 'text_ids', 'padding_mask', and 'language_masked_pos'

In [5]:
from collections import Counter

print(f"Loading raw annotation file from: {vqa_answer_label_path}")
with open(vqa_answer_label_path, 'r') as f:
    vqa_data = json.load(f)

# 'annotations' 키에 해당하는 리스트를 가져옵니다.
raw_annotations = vqa_data['annotations']
print(f"Found {len(raw_annotations)} annotations.")

# --- 2. 모든 답변들의 빈도를 계산합니다. ---
# Counter는 각 항목의 개수를 세어주는 유용한 도구입니다.
answer_counter = Counter()
for ann in raw_annotations:
    # VQAv2에서는 여러 답변 중 'multiple_choice_answer'가 최종 정답으로 간주됩니다.
    answer_counter[ann['multiple_choice_answer']] += 1

# --- 3. 가장 빈도수가 높은 3,129개의 답변을 선택합니다. ---
# 이것이 VQAv2의 표준 답변 후보 개수입니다.
num_top_answers = 3129
top_answers = answer_counter.most_common(num_top_answers)
answer_vocab = [answer for answer, count in top_answers]
print(f"Top {len(answer_vocab)} answers extracted.")

# --- 4. 최종적으로 사용할 answer_map (인덱스 -> 답변)을 생성합니다. ---
answer_map = {str(i): answer for i, answer in enumerate(answer_vocab)}
print("Successfully created the 'answer_map' dictionary!")

# --- 5. 생성된 answer_map 내용 일부 확인 (선택 사항) ---
print("\n--- Example of created answer_map ---")
for i in range(10):
    print(f"Index {i}: {answer_map[str(i)]}")

Loading raw annotation file from: /data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/v2_mscoco_train2014_annotations.json
Found 443757 annotations.
Top 3129 answers extracted.
Successfully created the 'answer_map' dictionary!

--- Example of created answer_map ---
Index 0: yes
Index 1: no
Index 2: 1
Index 3: 2
Index 4: white
Index 5: 3
Index 6: blue
Index 7: red
Index 8: black
Index 9: 0


In [6]:
transform = transforms.Compose([
    transforms.Resize((768, 768)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])
image = Image.open(image_path).convert("RGB")
image_tensor = transform(image).unsqueeze(0).to(device)

NameError: name 'image_path' is not defined

In [1]:
import os
from argparse import Namespace
from PIL import Image
import torch
from torchvision import transforms
import sentencepiece as spm

# 사용자가 제공한 모델 정의 코드를 그대로 사용하기 위해
# beit3_modeling.py 파일로 저장했다고 가정합니다.
# 만약 다른 파일 이름이라면 해당 이름으로 변경해주세요.
from modeling_finetune import beit3_large_patch16_480_captioning

# --------------------------------------------------------
# 1. 설정
# --------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 사용할 로컬 파일들의 이름
MODEL_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3_large_patch16_480_coco_captioning.pth"
TOKENIZER_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3.spm"

/data/2_data_server/cv-07/anaconda3/envs/beit3_env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class BEiT3Captioning:
    def __init__(self, model_path, tokenizer_path, device):
        self.device = device
        self.tokenizer = spm.SentencePieceProcessor()
        self.tokenizer.load(tokenizer_path)
        self.transform = transforms.Compose([
            transforms.Resize((480, 480), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
        ])
        
        model_kwargs = dict(
            vocab_size=64010, # 체크포인트에 맞게 어휘 크기 고정
            beam_size=1,
            max_len_b=16,
            drop_path_rate=0,
            layerscale_init_value=1e-5
        )
        
        print("모델 로드 중...")
        self.model = beit3_large_patch16_480_captioning(**model_kwargs)
        
        checkpoint = torch.load(model_path, map_location="cpu")
        self.model.load_state_dict(checkpoint['model'], strict=False)
        
        self.model.to(device)
        self.model.eval()
        print("모델 로드 완료.")

    def generate_caption(self, image_path, max_len=30):
        try:
            image = Image.open(image_path).convert("RGB")
        except FileNotFoundError:
            print(f"오류: 이미지 파일 '{image_path}'를 찾을 수 없습니다.")
            return None

        
        image_tensor = self.transform(image).unsqueeze(0).to(self.device)
        bos_token_id = self.tokenizer.bos_id()
        eos_token_id = self.tokenizer.eos_id()
        generated_ids = [bos_token_id]
        incremental_state = {}
        
        with torch.no_grad():
            for i in range(max_len):
                current_input_ids = torch.LongTensor([generated_ids]).to(self.device)
                image_input = image_tensor if i == 0 else None
                
                logits, incremental_state = self.model(
                    image=image_input,
                    text_ids=current_input_ids,
                    padding_mask=None,
                    language_masked_pos=None,
                    incremental_state=incremental_state
                )
                
                next_token_logits = logits[0, -1, :]
                next_token_id = torch.argmax(next_token_logits).item()

                if next_token_id == eos_token_id:
                    break
                
                generated_ids.append(next_token_id)

        caption = self.tokenizer.decode(generated_ids)
        return caption

In [3]:
# --- 사전 준비 ---
# 이 스크립트를 실행하기 전에, 아래 파일들이 모두 같은 폴더에 있는지 확인해주세요:
# 1. 이 파이썬 스크립트 파일
# 2. beit3_modeling.py
# 3. beit3_large_patch16_480_captioning.pth
# 4. tokenizer.model
# -----------------

try:
    # Step 1: 캡셔닝 모델 인스턴스 생성
    captioner = BEiT3Captioning(
        model_path=MODEL_PATH,
        tokenizer_path=TOKENIZER_PATH,
        device=DEVICE
    )

    # ===================================================================
    # Step 2: 여기에 이미지 파일 경로를 입력하세요
    # 예시:
    # user_image_path = "C:/Users/YourName/Pictures/my_photo.jpg"  (윈도우)
    # user_image_path = "/home/user/images/my_image.png"          (리눅스)
    # ===================================================================
    user_image_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images/TEST_000.jpg" # <--- 이 부분을 수정하세요
    
    if not os.path.exists(user_image_path):
            print("\n" + "="*50)
            print(f"오류: '{user_image_path}' 경로에 파일이 없습니다.")
            print("user_image_path 변수에 정확한 파일 경로를 입력해주세요.")
            print("="*50)
    else:
        print("\n" + "="*50)
        print(f"캡션을 생성할 이미지: '{user_image_path}'")
        
        generated_caption = captioner.generate_caption(user_image_path)
        
        if generated_caption:
            print(f"\n생성된 캡션:\n{generated_caption}")
        print("="*50)

except FileNotFoundError as e:
    print(f"\n오류: 필수 파일을 찾을 수 없습니다. '{e.filename}'")
    print("스크립트와 같은 폴더에 모델(.pth)과 토크나이저(.model) 파일이 있는지 확인해주세요.")
except Exception as e:
    print(f"알 수 없는 오류가 발생했습니다: {e}")

모델 로드 중...
모델 로드 완료.

캡션을 생성할 이미지: '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images/TEST_000.jpg'
알 수 없는 오류가 발생했습니다: The size of tensor a (902) must match the size of tensor b (904) at non-singleton dimension 2


In [4]:
import requests
from PIL import Image
import torch

# transformers 라이브러리가 없다면 설치해야 합니다.
try:
    from transformers import AutoProcessor, AutoModelForCausalLM
except ImportError:
    print("transformers 라이브러리를 설치합니다. 잠시만 기다려주세요.")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers"])
    from transformers import AutoProcessor, AutoModelForCausalLM


def generate_caption_with_huggingface(image_path):
    """
    Hugging Face transformers 라이브러리를 사용하여 이미지 캡션을 생성합니다.
    """
    try:
        image = Image.open(image_path).convert("RGB")
    except FileNotFoundError:
        print(f"오류: 이미지 파일 '{image_path}'를 찾을 수 없습니다.")
        return

    # Hugging Face Hub에서 BEiT3 모델과 프로세서를 자동으로 다운로드하고 로드합니다.
    print("Hugging Face에서 모델과 프로세서를 로드합니다. 처음 실행 시 시간이 걸릴 수 있습니다...")
    try:
        processor = AutoProcessor.from_pretrained("microsoft/beit3-large-patch16-480-captioning")
        model = AutoModelForCausalLM.from_pretrained("microsoft/beit3-large-patch16-480-captioning")
    except Exception as e:
        print(f"Hugging Face 모델 로드 중 오류 발생: {e}")
        print("인터넷 연결을 확인해주세요.")
        return

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    print("모델 로드 완료.")

    # 이미지 전처리 및 캡션 생성
    inputs = processor(images=image, text="", return_tensors="pt").to(device)

    print("\n캡션 생성 중...")
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_length=50)

    # 생성된 토큰을 텍스트로 변환
    generated_caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return generated_caption.strip()


if __name__ == "__main__":
    # 여기에 캡션을 생성할 이미지의 파일 경로를 입력하세요.
    user_image_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images/TEST_000.jpg"

    print("\n" + "="*50)
    print(f"캡션을 생성할 이미지: '{user_image_path}'")
    
    caption = generate_caption_with_huggingface(user_image_path)
    
    if caption:
        print(f"\n생성된 캡션:\n{caption}")
    
    print("="*50)


캡션을 생성할 이미지: '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images/TEST_000.jpg'
Hugging Face에서 모델과 프로세서를 로드합니다. 처음 실행 시 시간이 걸릴 수 있습니다...
Hugging Face 모델 로드 중 오류 발생: microsoft/beit3-large-patch16-480-captioning is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
인터넷 연결을 확인해주세요.
